# TD control: SARSA (on-policy) vs Q-learning (off-policy)

_Reinforcement Learning_

**One update rule learns the policy you follow; swap in a max and the same loop learns the optimal policy instead.**

You are dropped into a world with no map. You can only act, see the reward, and see where you
       land. TD control learns how to act anyway: it keeps a table $Q(s,a)$ of "how good is
       taking action $a$ in state $s$", and after every single step it nudges one entry toward what it
       just observed.

       The nudge is a bootstrap: the target uses $Q$'s own estimate of the next state. SARSA
       and Q-learning are the same loop with the same $\varepsilon$-greedy exploration. The only
       difference is which next-state value goes into the target &mdash; and that one choice decides
       whether you learn the policy you follow or the optimal policy.

---

This notebook is a practice scaffold for the **TD control: SARSA (on-policy) vs Q-learning (off-policy)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy + Gymnasium

We run the *same* TD-control loop twice on Gymnasium's **CliffWalking** grid — once as SARSA, once as Q-learning — and compare the greedy path each one learns. We build it in three steps: (1) set up the environment and the shared $\varepsilon$-greedy training loop, (2) read out the greedy path a trained $Q$-table implies, (3) run both methods and compare.

### Step 1 — Build the environment and the shared training loop

`CliffWalking-v0` is a 4×12 grid: start bottom-left, goal bottom-right, with a cliff along the bottom row between them. Every step costs `-1`; stepping into the cliff costs `-100` and snaps you back to the start. The `train` loop is identical for both methods — same $\varepsilon$-greedy exploration, same per-step update `Q[s,a] += alpha*(target - Q[s,a])`. The **only** difference is the bootstrap target: SARSA uses `Q[s2, a2]` (the action the policy *actually* takes next), while Q-learning uses `np.max(Q[s2])` (the best action available).

In [ ]:
import numpy as np
import gymnasium as gym

# CliffWalking-v0: 4x12 grid, start bottom-left, goal bottom-right, a cliff
# along the bottom row between them. Step reward -1; stepping into the cliff -100
# and reset to start. 48 states, 4 actions (up,right,down,left).
def train(method, episodes=500, alpha=0.5, gamma=1.0, eps=0.1, seed=0):
    env = gym.make("CliffWalking-v0")
    rng = np.random.default_rng(seed)
    nS = env.observation_space.n
    nA = env.action_space.n
    Q = np.zeros((nS, nA))

    def egreedy(s):
        # With probability eps explore a random action, otherwise act greedily.
        if rng.random() < eps:
            return rng.integers(nA)
        return int(np.argmax(Q[s]))

    for ep in range(episodes):
        s, _ = env.reset(seed=seed + ep)
        a = egreedy(s)
        done = False
        while not done:
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            a2 = egreedy(s2)                 # next action the policy WOULD take
            if method == "sarsa":            # ON-POLICY: value of the action actually taken
                bootstrap = 0 if done else gamma * Q[s2, a2]
            else:                            # Q-LEARNING, OFF-POLICY: value of the best action
                bootstrap = 0 if done else gamma * np.max(Q[s2])
            target = r + bootstrap
            Q[s, a] += alpha * (target - Q[s, a])
            s, a = s2, a2
    env.close()
    return Q

### Step 2 — Read out the greedy path a Q-table implies

Training fills in `Q`, but what we really want to see is the route the agent would take if it stopped exploring and acted greedily everywhere. `greedy_path` walks the environment taking `argmax(Q[s])` at each step and records the states it visits — that trajectory is the policy the table has learned.

In [ ]:
def greedy_path(Q, max_steps=50):
    env = gym.make("CliffWalking-v0")
    s, _ = env.reset(seed=0)
    path = [s]
    for _ in range(max_steps):
        a = int(np.argmax(Q[s]))   # always take the greedy action
        s, r, term, trunc, _ = env.step(a)
        path.append(s)
        if term or trunc:
            break
    env.close()
    return path

### Step 3 — Train both methods and compare their paths

Now run the same loop twice and read out each greedy path. Q-learning's path is **shorter**: its `max` target assumes optimal play next, so it learns to hug the cliff edge — the fewest steps. SARSA's path sits one row higher and is longer: its on-policy target bakes in the occasional exploratory misstep, so it prefers the safer route that survives $\varepsilon$-exploration.

In [ ]:
Q_sarsa = train("sarsa")
Q_qlearn = train("qlearning")

sarsa_path = greedy_path(Q_sarsa)
qlearn_path = greedy_path(Q_qlearn)

print("SARSA greedy path length:    ", len(sarsa_path))
print("Q-learning greedy path length:", len(qlearn_path))
print("SARSA path (states):    ", sarsa_path)
print("Q-learning path (states):", qlearn_path)
# Q-learning's path is shorter (hugs the cliff edge, the optimal route);
# SARSA's is one row higher and longer -- the safe route that survives epsilon-exploration.

## Visualize the data & results

_On a CliffWalking grid, how do SARSA's and Q-learning's average return per episode compare DURING training?_

We rebuild a small CliffWalking by hand (so there are no external dependencies) and track the *online* return each method earns while it is still exploring. We do it in three steps: (1) define the gridworld dynamics, (2) run both methods recording per-episode return, (3) smooth the curves and compare.

### Step 1 — Define the gridworld by hand

Here we spell out a tiny 4×6 CliffWalking-style grid directly in NumPy. `step` encodes the dynamics: move in the chosen direction (clamped to the grid), and if you land in a cliff cell you get `-100` and reset to the start; reaching the goal gives `-1` and ends the episode; any other move costs `-1`. `egreedy` is the same exploration rule as before.

In [ ]:
import numpy as np
np.random.seed(0)

# Tiny CliffWalking-like grid: 4 rows x 6 cols. Start = bottom-left (3,0),
# goal = bottom-right (3,5). Cliff = bottom row cells (3,1)..(3,4):
# stepping in gives reward -100 and resets to start. Every other step: -1.
ROWS, COLS = 4, 6
START, GOAL = (3, 0), (3, 5)
CLIFF = set((3, c) for c in range(1, 5))
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
nS, nA = ROWS * COLS, 4
sid = lambda r, c: r * COLS + c

def step(r, c, a):
    dr, dc = ACTIONS[a]
    nr = min(max(r + dr, 0), ROWS - 1)
    nc = min(max(c + dc, 0), COLS - 1)
    if (nr, nc) in CLIFF:
        return START[0], START[1], -100.0, False   # fell in: big penalty, back to start
    if (nr, nc) == GOAL:
        return nr, nc, -1.0, True                   # reached goal: episode ends
    return nr, nc, -1.0, False                      # ordinary step

def egreedy(Q, s, eps):
    if np.random.rand() < eps:
        return np.random.randint(nA)
    return int(np.argmax(Q[s]))

### Step 2 — Run both methods, recording the return each episode

`run` is the same TD-control loop, but now it accumulates the total reward `G` collected during each episode and returns the sequence of returns. Because the agent is acting $\varepsilon$-greedily throughout, `G` reflects its *online* behavior — falls off the cliff included — not the polished greedy path. SARSA and Q-learning again differ only in the target.

In [ ]:
def run(method, episodes=500, alpha=0.5, gamma=1.0, eps=0.1):
    Q = np.zeros((nS, nA))
    returns = []
    for ep in range(episodes):
        r, c = START
        s = sid(r, c)
        a = egreedy(Q, s, eps)
        G = 0.0
        done = False
        steps = 0
        while not done and steps < 200:
            nr, nc, rew, done = step(r, c, a)
            ns = sid(nr, nc)
            G += rew
            na = egreedy(Q, ns, eps)
            if method == "sarsa":                    # on-policy target
                bootstrap = 0 if done else gamma * Q[ns, na]
            else:                                    # off-policy target (the max)
                bootstrap = 0 if done else gamma * np.max(Q[ns])
            target = rew + bootstrap
            Q[s, a] += alpha * (target - Q[s, a])
            r, c = nr, nc
            s = ns
            a = na
            steps += 1
        returns.append(G)
    return np.array(returns)

### Step 3 — Smooth and compare the online returns

Raw per-episode returns are noisy, so we smooth each curve with a 20-episode moving average and sample a few checkpoints. The punchline: Q-learning's averaged return sits **lower** than SARSA's during training. Even though Q-learning learns the shorter optimal path, its exploratory steps near the cliff edge occasionally cost `-100`, dragging its online return below SARSA's safer detour.

In [ ]:
Rs = run("sarsa")
Rq = run("qlearning")

smooth = lambda x, w=20: np.convolve(x, np.ones(w) / w, mode="valid")
Ss = smooth(Rs)
Sq = smooth(Rq)

xs = list(range(20, 501, 40))
print("episode:   ", xs)
print("SARSA:     ", [round(float(Ss[x - 20]), 1) for x in xs])
print("Q-learning:", [round(float(Sq[x - 20]), 1) for x in xs])
print("mean last 300 -> SARSA", round(float(Rs[200:].mean()), 1),
      "| Q-learning", round(float(Rq[200:].mean()), 1))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In state $S'$ the action-values are $Q(S',\text{up})=4$, $Q(S',\text{down})=1$. The agent's $\varepsilon$-greedy roll explores and takes $A'=\text{down}$. With $R=0$, $\gamma=0.9$, write the bootstrap term used by SARSA and by Q-learning.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- SARSA uses the action actually taken: $A'=\text{down}$, so $\gamma Q(S',A') = 0.9\times 1 = 0.9$. — _On-policy: the target reflects the policy being followed, exploration and all._
- Q-learning uses the max: $\gamma\max_{a'}Q(S',a') = 0.9\times 4 = 3.6$. — _Off-policy: the target reflects the greedy/optimal policy, ignoring the explored action._

**Answer:** SARSA bootstrap $=0.9$; Q-learning bootstrap $=3.6$. The gap is exactly $Q(S',A')$ vs $\max_{a'}Q(S',a')$.

</details>

**Problem 2.** On CliffWalking, why does Q-learning's average return DURING training come out lower than SARSA's, even though Q-learning learns the shorter optimal path?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Q-learning learns the optimal greedy path: hug the cliff edge for the fewest steps. — _Its max target assumes optimal play next, so cliff-edge states look best._
- But it still acts $\varepsilon$-greedily, so it occasionally takes a random step right off the cliff (reward $-100$). — _The behavior policy explores; the cliff-edge path has no safety margin for those random steps._
- SARSA's on-policy target bakes in those exploratory falls, so it values cliff-adjacent states lower and learns a safer detour. — _Accounting for its own exploration, it trades a few extra steps for not falling._

**Answer:** Q-learning's risky-during-training behavior (optimal path + exploration = frequent cliff falls) lowers its online return below SARSA's safer path, even though its learned greedy path is shorter.

</details>

**Problem 3.** Define on-policy and off-policy in one sentence each, and say which method is which.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- On-policy: the policy being learned is the same policy used to generate the data. — _SARSA learns $Q^{\pi}$ for its own $\varepsilon$-greedy $\pi$._
- Off-policy: the policy being learned (target) differs from the behavior policy generating the data. — _Q-learning learns the greedy $Q^{*}$ while behaving $\varepsilon$-greedily._

**Answer:** SARSA is on-policy (learns the followed policy via $Q(S',A')$); Q-learning is off-policy (learns the optimal policy via $\max_{a'}Q(S',a')$ regardless of behavior).

</details>